In [5]:
from google.colab import drive
drive.mount('/content/drive')

import cv2
import numpy as np
import matplotlib.pyplot as plt


"""
Algoritimo White Patch com Threshold (Percentil).
Ignora os X% pixels mais claros e mais escuros para tentar evitar ruído.
"""
def white_patch_retinex_com_Threshold(img, threshold=0.01):

    img = img.astype(np.float32)

    # Separa canais (OpenCV usa BGR)
    channels = cv2.split(img)
    out_channels = []

    # Calcula limites baseados em porcentagem (ex: 1% e 99%)
    # np.percentile é muito otimizado
    lower_p = threshold * 100
    upper_p = (1 - threshold) * 100

    for channel in channels:
        # Encontra os valores de corte (min e max) baseados no percentil
        v_min, v_max = np.percentile(channel, [lower_p, upper_p])

        # Evita divisão por zero
        if v_max == v_min:
            out_channels.append(channel)
            continue

        # Transformação Linear Vetorizada (Substitui os loops 'for')
        # Fórmula: (pixel - min) * (255 / (max - min))
        scale = 255.0 / (v_max - v_min)
        channel_new = (channel - v_min) * scale

        # Garante que nada passe de 0 ou 255 (Clip)
        channel_new = np.clip(channel_new, 0, 255)
        out_channels.append(channel_new)

    return cv2.merge(out_channels).astype(np.uint8)



"""
Algoritimo Gray World com clippagem final baseada em threshold.
"""
def gray_world_com_Threshold(img, threshold=0.01):
    img = img.astype(np.float32)
    b, g, r = cv2.split(img)

    # 1. Correção das Médias (Gray World Padrão)
    mean_b = np.mean(b)
    mean_g = np.mean(g)
    mean_r = np.mean(r)

    mean_gray = (mean_b + mean_g + mean_r) / 3

    # Fatores de escala para neutralizar as cores
    # (Evita divisão por zero com np.maximum)
    scale_b = mean_gray / np.maximum(mean_b, 1e-5)
    scale_g = mean_gray / np.maximum(mean_g, 1e-5)
    scale_r = mean_gray / np.maximum(mean_r, 1e-5)

    b *= scale_b
    g *= scale_g
    r *= scale_r

    # 2. Normalização Robusta (Igual ao seu código, mas vetorizado)
    # Combinamos os canais temporariamente para achar o corte global
    merged = cv2.merge([b, g, r])

    # Acha o valor que corta os X% mais brilhantes da imagem corrigida
    upper_p = (1 - threshold) * 100
    v_max = np.percentile(merged, upper_p)

    if v_max == 0: v_max = 1 # Segurança

    # Escala tudo para que esse "quase máximo" vire 255
    scale_global = 255.0 / v_max
    merged = merged * scale_global

    return np.clip(merged, 0, 255).astype(np.uint8)

# EXECUÇÃO DO PROGRAMA

#Carregasse as imagens
filenames = ['/content/drive/MyDrive/Pesquisa/Visão Computacional/imagens/Quest5/Buildings.png',
             '/content/drive/MyDrive/Pesquisa/Visão Computacional/imagens/Quest5/Church.png',
             '/content/drive/MyDrive/Pesquisa/Visão Computacional/imagens/Quest5/ColorChecker.png',
             '/content/drive/MyDrive/Pesquisa/Visão Computacional/imagens/Quest5/toysflash.png',
             '/content/drive/MyDrive/Pesquisa/Visão Computacional/imagens/Quest5/toysnoflash.png']

# Threshold de 1% (0.01) é o padrão da indústria
# Variouse o Threshold comentando e descomentando o threshold_val desejado
#threshold_val = 0.01
#threshold_val = 0.13
threshold_val = 0.25

plt.figure(figsize=(15, 20))

for i, f in enumerate(filenames):
    img_bgr = cv2.imread(f)
    if img_bgr is None: continue

    # Processamento, chama as funções que executam os algoritimos
    res_wp = white_patch_retinex_com_Threshold(img_bgr, threshold=threshold_val)
    res_gw = gray_world_com_Threshold(img_bgr, threshold=threshold_val)

    # Converter p/ RGB
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    res_wp_rgb = cv2.cvtColor(res_wp, cv2.COLOR_BGR2RGB)
    res_gw_rgb = cv2.cvtColor(res_gw, cv2.COLOR_BGR2RGB)

    # Plots
    plt.subplot(len(filenames), 3, i*3 + 1)
    plt.imshow(img_rgb)
    plt.title("Original")
    plt.axis('off')

    plt.subplot(len(filenames), 3, i*3 + 2)
    plt.imshow(res_wp_rgb)
    plt.title(f"White Patch (Thresh {threshold_val})")
    plt.axis('off')

    plt.subplot(len(filenames), 3, i*3 + 3)
    plt.imshow(res_gw_rgb)
    plt.title(f"Gray World (Thresh {threshold_val})")
    plt.axis('off')

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.